# SC-21-Move-Sui - Move sur Sui

[<< Bitcoin Scripting](SC-20-Bitcoin-Scripting.ipynb) | [Solana & Anchor >>](SC-22-Solana-Anchor.ipynb)

---

## Objectifs d'apprentissage

1. Decouvrir le langage **Move** (Meta/Diem)
2. Comprendre le **modele objet** de Sui
3. Creer un module Move simple
4. Utiliser les **abilities** (key, store, copy, drop)

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-4](../01-Solidity-Foundation/SC-6-Errors-Events.ipynb) completes (concepts)
- Notions de programmation orientee ressources
- Sui CLI installe (voir instructions dans le notebook)

### Duree estimee : 50 minutes

---

## 1. Introduction a Move

Move est un langage concu pour les smart contracts avec une semantique de ressources.

In [1]:
# Comparaison Move vs Solidity
print("""
MOVE vs SOLIDITY

| Caracteristique    | Solidity         | Move                |
|-------------------|------------------|---------------------|
| Paradigme         | Comptes          | Objets              |
| Ressources        | Non-lineaires    | Lineaires (assets)  |
| Ownership         | Implicit         | Explicit (abilities)|
| Generique         | Non              | Oui                 |
| Bytecode          | EVM              | Move VM             |

ABILITIES (Capacites):
- key    : Peut etre un identifiant global
- store  : Peut etre stocke dans un objet
- copy   : Peut etre copie
- drop   : Peut etre detruit implicitement
""")


MOVE vs SOLIDITY

| Caracteristique    | Solidity         | Move                |
|-------------------|------------------|---------------------|
| Paradigme         | Comptes          | Objets              |
| Ressources        | Non-lineaires    | Lineaires (assets)  |
| Ownership         | Implicit         | Explicit (abilities)|
| Generique         | Non              | Oui                 |
| Bytecode          | EVM              | Move VM             |

ABILITIES (Capacites):
- key    : Peut etre un identifiant global
- store  : Peut etre stocke dans un objet
- copy   : Peut etre copie
- drop   : Peut etre detruit implicitement



---

## 2. Module Move Simple

Exemple d'un token simple sur Sui.

In [2]:
# Module Move: Coin natif
MOVE_COIN = '''
/// Module: my_coin.move
module my_package::my_coin {
    use sui::coin::{Self, Coin};
    use sui::transfer;
    use sui::object::{Self, UID};
    use sui::tx_context::{Self, TxContext};

    /// Erreur: montant insuffisant
    const EInsufficientBalance: u64 = 0;

    /// Structure du token (capability pour mint)
    public struct AdminCap has key {
        id: UID,
    }

    /// One-Time Witness pour le coin
    public struct MY_COIN has drop {}

    /// Initialisation du module
    fun init(witness: MY_COIN, ctx: &mut TxContext) {
        // Creer la capability admin
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );

        // Enregistrer le coin
        coin::create_currency(witness, 9, b"MYC", b"My Coin", b"", ctx);
    }

    /// Minter de nouveaux tokens
    public fun mint(
        _: &AdminCap,
        amount: u64,
        recipient: address,
        ctx: &mut TxContext
    ) {
        let coin = coin::mint(amount, ctx);
        transfer::public_transfer(coin, recipient);
    }
}
'''

print("Module Move - Coin:")
print(MOVE_COIN)

Module Move - Coin:

/// Module: my_coin.move
module my_package::my_coin {
    use sui::coin::{Self, Coin};
    use sui::transfer;
    use sui::object::{Self, UID};
    use sui::tx_context::{Self, TxContext};

    /// Erreur: montant insuffisant
    const EInsufficientBalance: u64 = 0;

    /// Structure du token (capability pour mint)
    public struct AdminCap has key {
        id: UID,
    }

    /// One-Time Witness pour le coin
    public struct MY_COIN has drop {}

    /// Initialisation du module
    fun init(witness: MY_COIN, ctx: &mut TxContext) {
        // Creer la capability admin
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );

        // Enregistrer le coin
        coin::create_currency(witness, 9, b"MYC", b"My Coin", b"", ctx);
    }

    /// Minter de nouveaux tokens
    public fun mint(
        _: &AdminCap,
        amount: u64,
        recipient: address,
        ctx: &mut TxContext
    ) {
    

["### Interpretation : Move vs Solidity - Paradigmes Different\n", "\n", "Ce tableau compare les differents paradigmes de programmation entre Solidity (EVM) et Move (Sui/Aptos).\n", "\n", "| Caracteristique | Solidity (EVM) | Move (Sui) | Impact pratique |\n", "|----------------|---------------|-----------|-----------------|\n", "| **Paradigme** | Comptes (account-based) | Objets (object-based) | Solidity : etat global dans le contrat; Move : objets independants avec proprietaires |\n", "| **Ressources** | Non-lineaires (copiables) | Lineaires (uniques) | Move : impossible de copier un asset sans autorisation explicite |\n", "| **Ownership** | Implicite (mapping address → uint) | Explicite (abilities `key`/`store`) | Move : le compilateur verifie qui peut posseder quoi |\n", "| **Generiques** | Non | Oui (types parametrables) | Move : `Coin<T>` est un type generique (fonctionne pour n'importe quel token) |\n", "| **Bytecode** | EVM (stack-based) | Move VM (registres, bytecodes) | Move : plus proche de machine native, optimisations possibles |\n", "\n", "**Points cles** :\n", "1. **Abilities (capacites)** : En Move, chaque struct declare explicitement ce qu'elle peut faire (`key`, `store`, `copy`, `drop`)\n", "2. **Ressources lineaires** : Un token `Coin` est une ressource qui ne peut pas etre copiee (pas de double-spend possible)\n", "3. **Modele objet** : Sur Sui, chaque NFT, chaque token est un objet avec son propre UID (pas d'index dans un mapping)\n", "4. **Parallelisme** : Les objets Move peuvent etre modifies en parallele s'ils sont independants (pas de contention globale)\n", "\n", "**Exemples d'abilities** :\n", "- `key` : L'objet peut etre stocke globalement (possede par une adresse)\n", "- `store` : L'objet peut etre stocke dans un autre objet (nesting)\n", "- `copy` : L'objet peut etre copie (par defaut : non, pour securite)\n", "- `drop` : L'objet peut etre detruit implicitement (sinon : destruction manuelle obligatoire)\n", "\n", "**Implications pour les developpeurs** :\n", "- Sur Ethereum : on deploye un contrat avec des mappings globaux (balances, allowances)\n", "- Sur Sui : on deploie un module qui definit des types d'objets (NFTs, coins, escrows)\n", "- Le modele Move est plus verifiable par le compilateur (moins de bugs de runtime)\n", "\n", "> **Note technique** : Le terme \"lineaire\" en Move signifie que la valeur ne peut etre utilisee qu'une seule fois. Quand une fonction consomme une ressource lineaire, elle doit la \"detruire\" ou la \"transferer\".\n"]

---

## 3. Modele Objet Sui

Chaque objet a un proprietaire unique.

In [3]:
# NFT simple sur Sui
MOVE_NFT = '''
module my_package::my_nft {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::tx_context::{Self, TxContext};
    use sui::url::Url;

    /// Structure NFT
    public struct NFT has key, store {
        id: UID,
        name: String,
        description: String,
        url: Url,
    }

    /// Admin capability pour minter
    public struct AdminCap has key { id: UID }

    /// Initialisation
    fun init(ctx: &mut TxContext) {
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );
    }

    /// Minter un nouveau NFT
    public fun mint(
        _: &AdminCap,
        name: String,
        description: String,
        url: Url,
        recipient: address,
        ctx: &mut TxContext
    ) {
        let nft = NFT {
            id: object::new(ctx),
            name,
            description,
            url,
        };
        transfer::public_transfer(nft, recipient);
    }

    /// Transferer un NFT
    public fun transfer(nft: NFT, recipient: address) {
        transfer::public_transfer(nft, recipient);
    }

    /// Burn un NFT
    public fun burn(nft: NFT) {
        let NFT { id, name: _, description: _, url: _ } = nft;
        object::delete(id);
    }
}
'''

print("Module Move - NFT:")
print(MOVE_NFT)

Module Move - NFT:

module my_package::my_nft {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::tx_context::{Self, TxContext};
    use sui::url::Url;

    /// Structure NFT
    public struct NFT has key, store {
        id: UID,
        name: String,
        description: String,
        url: Url,
    }

    /// Admin capability pour minter
    public struct AdminCap has key { id: UID }

    /// Initialisation
    fun init(ctx: &mut TxContext) {
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );
    }

    /// Minter un nouveau NFT
    public fun mint(
        _: &AdminCap,
        name: String,
        description: String,
        url: Url,
        recipient: address,
        ctx: &mut TxContext
    ) {
        let nft = NFT {
            id: object::new(ctx),
            name,
            description,
            url,
        };
        transfer::public_transfer(nft, recipient);
    }

    ///

["### Interpretation : Module Move - Coin Natif Sui\n", "\n", "Ce module presente la creation d'un token natif sur Sui en utilisant le systeme de coin built-in.\n", "\n", "| Composant | Role | Details |\n", "|-----------|------|---------|\n", "| `AdminCap` | Capability de mintage | Objet `key` transférable, represente l'autorite de mint |\n", "| `MY_COIN` | One-Time Witness | Type avec `drop` qui n'existe qu'en un seul exemplaire (pour `coin::create_currency`) |\n", "| `init()` | Initialisation du module | Appelé automatiquement a la premiere publication, cree la capability et enregistre le coin |\n", "| `mint()` | Creation de tokens | Consomme la capability, mint des coins vers un recipient |\n", "\n", "**Points cles** :\n", "1. **One-Time Witness (OTW)** : `MY_COIN` est un type qui ne peut etre instancié qu'une seule fois (garanti par le compilateur Move)\n", "2. `coin::create_currency(witness, 9, b\"MYC\", ...)` : enregistre le coin avec 9 decimales, symbole \"MYC\"\n", "3. `coin::mint(amount, ctx)` : cree `amount` tokens du coin (utilise le witness enregistre)\n", "4. `transfer::public_transfer(coin, recipient)` : transfere les coins nouvellement mints vers le destinataire\n", "\n", "**Abilities Move** :\n", "- `key` : l'objet peut etre un identifiant global (adresse, objet own)\n", "- `store` : l'objet peut etre stocke dans un autre objet (nesting)\n", "- `copy` : l'objet peut etre copie (par defaut, les structs ne peuvent pas etre copiees)\n", "- `drop` : l'objet peut etre detruit implicitement (sinon, doit etre explicitement detruit)\n", "\n", "**Difference avec Solidity** :\n", "- **Solidity** : ERC-20 est un contrat avec `mapping(address => uint256) balances`\n", "- **Move** : Coin est un objet lineaire (ressource) qui existe en un seul exemplaire par unite\n", "\n", "**Avantages du modele Move** :\n", "- **Lineaire** : les coins ne peuvent pas etre copies ou doubles (garanti par le type system)\n", "- **Pas de double-spend** : impossible de depenser les memes coins deux fois (le compilateur l'interdit)\n", "- **Performance** : les objects sont des ressources uniques (pas de mapping global a scanner)\n", "\n", "> **Note technique** : L'OTW (One-Time Witness) est un pattern unique a Move. Le type porte le meme nom que le module (`MY_COIN` dans `my_coin`), ce qui permet au compilateur de garantir l'unicite.\n"]

---

## 4. Commandes Sui CLI

Compilation et deploiement.

In [4]:
# Commandes Sui CLI
print("""
INSTALLATION:
cargo install --locked --git https://github.com/MystenLabs/sui.git --branch mainnet sui

CONFIGURATION:
sui client active-address     # Voir l'adresse active
sui client active-env        # Voir l'environnement
sui client switch --env devnet  # Changer d'environnement

PROJET:
sui move new my_package    # Creer un nouveau projet
cd my_package
mkdir -p sources tests

COMPILATION:
sui move build            # Compiler le module

TESTS:
sui move test             # Executer les tests unitaires

DEPLOIEMENT:
sui client publish --gas-budget 100000000

APPELS:
sui client call --package 0x... --module my_coin \
    --function mint --args 1000 0x... --gas-budget 10000000
""")


INSTALLATION:
cargo install --locked --git https://github.com/MystenLabs/sui.git --branch mainnet sui

CONFIGURATION:
sui client active-address     # Voir l'adresse active
sui client active-env        # Voir l'environnement
sui client switch --env devnet  # Changer d'environnement

PROJET:
sui move new my_package    # Creer un nouveau projet
cd my_package
mkdir -p sources tests

COMPILATION:
sui move build            # Compiler le module

TESTS:
sui move test             # Executer les tests unitaires

DEPLOIEMENT:
sui client publish --gas-budget 100000000

APPELS:
sui client call --package 0x... --module my_coin     --function mint --args 1000 0x... --gas-budget 10000000



["### Interpretation : NFT sur Sui avec Move\n", "\n", "Ce module montre comment creer et gerer un NFT sur Sui en utilisant le modele objet de Move.\n", "\n", "| Composant | Role | Abilities |\n", "|-----------|------|-----------|\n", "| `struct NFT` | Representation du NFT | `key` (objet global), `store` (stockable dans d'autres objets) |\n", "| `struct AdminCap` | Capability pour minter | `key` (objet transférable, represente l'autorite) |\n", "| `mint()` | Creation d'un NFT | Consomme la capability, cree un objet NFT |\n", "| `transfer()` | Transfert de propriete | Deplace l'objet NFT vers un nouveau proprietaire |\n", "| `burn()` | Destruction du NFT | Supprime l'objet et libere l'UID |\n", "\n", "**Points cles** :\n", "1. `has key, store` : le NFT peut etre un objet global (proprietaire = adresse) et etre stocke dans d'autres objets\n", "2. `object::new(ctx)` : cree un nouvel objet avec un UID unique (identifiant global sur Sui)\n", "3. `transfer::public_transfer(nft, recipient)` : transfere la propriete de l'objet (atomic, sans approve prealable)\n", "4. `burn()` utilise le pattern \"destructuring\" : `let NFT { id, name: _, ... } = nft;` extrait l'UID pour suppression\n", "\n", "**Difference Solidity vs Move** :\n", "- **Solidity** : `mapping(uint256 => address) ownerOf;` (le NFT est un index dans un mapping)\n", "- **Move** : Le NFT est un objet avec son propre UID, possede par une adresse (pas de mapping global)\n", "\n", "**Avantages du modele Move** :\n", "- **Pas d'approbation prealable** : le proprietaire peut transferer directement (pas de `approve`/`setApprovalForAll`)\n", "- **Typage statique** : le compilateur verifie que les ressources ne sont pas copiees ou detruites par erreur\n", "- **Parallelisme** : chaque objet peut etre modifie independamment (pas de contention globale)\n", "\n", "> **Note technique** : L'ability `store` permet au NFT d'etre stocke dans d'autres objets (ex: une collection contient plusieurs NFTs). Sans `store`, le NFT ne peut etre possede que par une adresse.\n"]

---

## 5. Exemples guidés

In [5]:
# Exercice: Escrow simple
EXERCISE_ESCROW = '''
module my_package::escrow {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::coin::Coin;
    use sui::sui::SUI;
    use sui::tx_context::{Self, TxContext};

    /// Erreurs
    const ENotSeller: u64 = 0;
    const ENotCompleted: u64 = 1;

    /// Escrow object
    public struct Escrow has key {
        id: UID,
        seller: address,
        buyer: address,
        amount: u64,
        completed: bool,
    }

    /// Creer un escrow
    public fun create(
        seller: address,
        buyer: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ): Escrow {
        Escrow {
            id: object::new(ctx),
            seller,
            buyer,
            amount: coin::value(&payment),
            completed: false,
        }
    }

    /// Completer l'escrow
    public fun complete(
        escrow: &mut Escrow,
        sender: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ) {
        assert!(sender == escrow.buyer, ENotSeller);
        escrow.completed = true;
        transfer::public_transfer(payment, escrow.seller);
    }
}
'''
print("Exercice Escrow:")
print(EXERCISE_ESCROW)

Exercice Escrow:

module my_package::escrow {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::coin::Coin;
    use sui::sui::SUI;
    use sui::tx_context::{Self, TxContext};

    /// Erreurs
    const ENotSeller: u64 = 0;
    const ENotCompleted: u64 = 1;

    /// Escrow object
    public struct Escrow has key {
        id: UID,
        seller: address,
        buyer: address,
        amount: u64,
        completed: bool,
    }

    /// Creer un escrow
    public fun create(
        seller: address,
        buyer: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ): Escrow {
        Escrow {
            id: object::new(ctx),
            seller,
            buyer,
            amount: coin::value(&payment),
            completed: false,
        }
    }

    /// Completer l'escrow
    public fun complete(
        escrow: &mut Escrow,
        sender: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ) {
        assert!(sender == es

["### Interpretation : Commandes Sui CLI\n", "\n", "Ces commandes couvrent le cycle de vie complet de developpement d'un module Move sur Sui.\n", "\n", "| Commande | Action | Usage typique |\n", "|----------|--------|---------------|\n", "| `sui move new my_package` | Creer un nouveau projet | Initialise la structure de base (sources/, tests/, Move.toml) |\n", "| `sui move build` | Compiler le module Move | Verifier la syntaxe et les types |\n", "| `sui move test` | Executer les tests unitaires | Validation de la logique avant deploiement |\n", "| `sui client publish` | Deploier le module sur Sui | Publication du bytecode sur devnet/mainnet |\n", "| `sui client call` | Appeler une fonction publique | Interaction avec le module deploye |\n", "\n", "**Points cles** :\n", "1. `--gas-budget` est obligatoire sur Sui pour limiter le cout maximum d'une transaction (en Mist = 10^-9 SUI)\n", "2. `sui client active-address` montre l'adresse actuellement utilisee pour signer les transactions\n", "3. `sui client switch --env devnet` change entre devnet, testnet et mainnet (config dans ~/.sui/sui_config/client.yaml)\n", "4. L'adresse du package (`0x...`) retournee par `publish` est necessaire pour les appels ulterieurs\n", "\n", "**Workflow typique** :\n", "```\n", "sui move new mon_nft      # Creer le projet\n", "# Editer sources/mon_nft.move\n", "sui move build            # Compiler (verifier la syntaxe)\n", "sui move test             # Tester (unite tests)\n", "sui client publish --gas-budget 100000000  # Deploier\n", "sui client call --package <addr> --module mon_nft --function mint --args ...  # Appeler\n", "```\n", "\n", "**Differences avec Solidity** :\n", "- Sui n'a pas de \"deployement de contrat\" classique : on publie un package avec plusieurs modules\n", "- Les modules ne sont pas deployes separement : tout le package est deploie en une transaction\n", "- Le package ID est l'identifiant unique de toutes les fonctions du module\n", "\n", "> **Note technique** : Le gas budget est en Mist (1 SUI = 1,000,000,000 Mist). Un budget de 100,000,000 Mist = 0.1 SUI. Si la transaction coute moins, la difference est remboursee.\n"]

---

## 6. Resume

| Concept | Description |
|---------|-------------|
| `module` | Unite de code Move |
| `struct` | Structure de donnees avec abilities |
| `key` | Peut etre un objet global |
| `store` | Peut etre stocke dans un autre objet |
| `UID` | Identifiant unique d'objet |
| `TxContext` | Contexte de transaction |

---

**Notebook suivant** : [SC-22-Solana-Anchor](SC-22-Solana-Anchor.ipynb)

---

[<< Bitcoin Scripting](SC-20-Bitcoin-Scripting.ipynb) | [Solana & Anchor >>](SC-22-Solana-Anchor.ipynb)

["### Interpretation : Exercice Escrow Move\n", "\n", "Cet exercice presente un escrow (tiers de confiance) simple sur Sui avec le modele objet de Move.\n", "\n", "| Fonction | Operation | Verification |\n", "|----------|-----------|--------------|\n", "| `create` | Cree un objet escrow avec paiement du vendeur | Stocke `seller`, `buyer`, `amount`, `completed=false` |\n", "| `complete` | Le buyer complete l'escrow et transfere le paiement | Verifie que `sender == buyer`, marke `completed=true` |\n", "\n", "**Points cles** :\n", "1. L'objet `Escrow` a l'ability `key` : il peut etre stocke comme un objet global (pas dans un compte)\n", "2. Le paiement `Coin<SUI>` est un token lineaire (ressource) : il ne peut pas etre copie, seulement transfere\n", "3. `assert!(sender == escrow.buyer, ENotSeller)` : verifie que seul le buyer peut completer l'escrow\n", "4. `transfer::public_transfer(payment, escrow.seller)` : transfere le paiement au vendeur (atomic avec la completion)\n", "\n", "**Limitation de l'exercice** :\n", "- La fonction `create` ne consomme pas le paiement du vendeur (devrait etre transferre dans l'escrow)\n", "- L'erreur `ENotCompleted` est definie mais jamais utilisee\n", "- Il manque une fonction `cancel` pour que le vendeur puisse annuler et recuperer ses fonds\n", "- L'objet escrow devrait etre detruit (`object::delete`) apres completion pour nettoyer\n", "\n", "**Ameliorations possibles** :\n", "- Ajouter un timeout pour annulation automatique\n", "- Permettre au vendeur de recuperer les fonds si le buyer ne complete pas\n", "- Detruire l'objet escrow apres completion pour liberer l'UID\n", "\n", "> **Note technique** : Sur Sui, les objets avec `key` ability sont des \"objets owns\". Ils ont un proprietaire unique (adresse ou autre objet). Quand un objet est transfere, la propriete change instantanement.\n"]